## Dataset tutorial (Zarr): minimal end-to-end walkthrough

This notebook shows the shortest path from **writing one synthetic episode** to **loading training batches**.

Sections:

- **ZarrWriter**: write a single `.zarr` episode (we reuse it throughout)
- **ZarrEpisode**: read raw keys / ranges
- **ZarrDataset**: turn episode storage keys into model-facing batch keys (with horizons)
- **MultiDataset**: combine datasets into one index space
- **from local resolver**: load episodes from a local folder with filters
- **from s3 resolver**: example code (disabled by default)


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np

from egomimic.rldb.zarr.zarr_dataset_multi import (
    LocalEpisodeResolver,
    MultiDataset,
    S3EpisodeResolver,
    ZarrDataset,
    ZarrEpisode,
)
from egomimic.rldb.zarr.zarr_writer import ZarrWriter

# Notebook-local output directory (you can change this)
OUT_DIR = Path(os.environ.get("EGOVERSE_TUTORIAL_OUT", "./data/tutorial_zarr")).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

EP_NAME = "demo_episode_000000"
EP_PATH = OUT_DIR / f"{EP_NAME}.zarr"

print("OUT_DIR:", OUT_DIR)
print("EP_PATH:", EP_PATH)

## ZarrWriter

We write **one** minimal episode and reuse it for the rest of the notebook.

We’ll store:

- one JPEG image key: `images.front_1`
- one numeric observation key: `state.proprio`
- one numeric action key: `actions.cartesian`
- language spans under `annotations`

The writer also writes episode metadata under `attrs` (including `embodiment`, `total_frames`, and a `features` schema).

In [ ]:
def write_synthetic_episode(*, episode_path: Path, T: int = 40, H: int = 120, W: int = 160) -> Path:
    # Images: (T,H,W,3) uint8
    base = np.linspace(0, 255, num=W, dtype=np.uint8)[None, None, :, None]
    img = np.repeat(base, H, axis=1)
    img = np.repeat(img, 3, axis=3)
    images = np.repeat(img, T, axis=0)

    # Numeric observation: (T, D)
    proprio = np.random.randn(T, 8).astype(np.float32)

    # Numeric action: (T, A)
    actions = np.random.randn(T, 7).astype(np.float32)

    annotations = [
        ("reach", 0, max(0, T // 2 - 1)),
        ("place", T // 2, T - 1),
    ]

    ZarrWriter.create_and_write(
        episode_path=episode_path,
        numeric_data={
            "state.proprio": proprio,
            "actions.cartesian": actions,
        },
        image_data={
            "images.front_1": images,
        },
        embodiment="tutorial_demo",
        fps=30,
        task="synthetic_demo",
        annotations=annotations,
        chunk_timesteps=50,
        enable_sharding=False,
    )

    return episode_path


_ = write_synthetic_episode(episode_path=EP_PATH)
print("Wrote episode:", EP_PATH)

## ZarrEpisode

`ZarrEpisode` is a thin reader around the `.zarr` folder.

- `metadata` comes from `group.attrs`
- `read({key: (start, end)})` reads either a **single frame** (`end=None`) or a **range** (`end` exclusive)


In [ ]:
ep = ZarrEpisode(EP_PATH)
print("metadata keys:", sorted(ep.metadata.keys()))
print("embodiment:", ep.metadata.get("embodiment"))
print("total_frames:", ep.metadata.get("total_frames"))

# Read one frame of numeric data
one = ep.read({"state.proprio": (0, None)})["state.proprio"]
print("state.proprio[0]:", one.shape, one.dtype)

# Read a range of actions
rng = ep.read({"actions.cartesian": (0, 5)})["actions.cartesian"]
print("actions.cartesian[0:5]:", rng.shape, rng.dtype)

# Images are stored as JPEG bytes (decoded later by ZarrDataset)
img0_bytes = ep.read({"images.front_1": (0, None)})["images.front_1"]
print("images.front_1[0] type:", type(img0_bytes), "bytes=", len(img0_bytes))

## ZarrDataset

`ZarrDataset` maps **dataset output keys** (what your model sees) to **episode store keys** (what’s inside the Zarr).

A minimal `key_map` entry looks like:

- output key: e.g. `observations.state.proprio`
- mapping: `{ "zarr_key": "state.proprio", "horizon": optional }`

Notes:

- Add `horizon` to read a **sequence** starting at `idx`.
- Near the end of the episode, sequences are padded (repeat last element) so shapes stay consistent.
- JPEG decode happens inside `ZarrDataset` for single-frame image reads (no horizon).

In [ ]:
key_map = {
    # images: single frame -> (C,H,W) float32
    "observations.images.front_img_1": {"zarr_key": "images.front_1"},
    # proprio: single frame -> (D,)
    "observations.state.proprio": {"zarr_key": "state.proprio"},
    # action chunk: horizon -> (H,A)
    "actions.cartesian": {"zarr_key": "actions.cartesian", "horizon": 10},
    # language annotations span -> string per frame (dataset-only special case)
    "language": {"zarr_key": "annotations"},
}

zds = ZarrDataset(Episode_path=EP_PATH, key_map=key_map, transform_list=None)

idx = len(zds) - 2  # near the end to show padding
sample = zds[idx]

print("idx:", idx, "/", len(zds))
print("keys:", sorted(sample.keys()))

print("\nobservations.images.front_img_1:", sample["observations.images.front_img_1"].shape)
print("observations.state.proprio:", sample["observations.state.proprio"].shape)
print("actions.cartesian:", sample["actions.cartesian"].shape)
print("language:", repr(sample["language"]))

## MultiDataset

`MultiDataset` stitches multiple datasets into a single index space and adds metadata like `embodiment`.

Here we create a `MultiDataset` from two copies of the same episode (just to show behavior).

In [ ]:
ds_a = ZarrDataset(Episode_path=EP_PATH, key_map=key_map, transform_list=None)
ds_b = ZarrDataset(Episode_path=EP_PATH, key_map=key_map, transform_list=None)

multi = MultiDataset(datasets={"a": ds_a, "b": ds_b}, mode="total")

print("len(single):", len(ds_a))
print("len(multi):", len(multi))

m0 = multi[0]
print("multi[0] has embodiment:", m0.get("embodiment"))
print("multi[0] metadata.robot_name:", m0.get("metadata.robot_name"))

## from local resolver

`LocalEpisodeResolver` scans a folder of `.zarr` episodes and filters them using each episode’s metadata.

We’ll point it at the folder we just wrote to and create a `MultiDataset` via `MultiDataset._from_resolver(...)`.

In [ ]:
resolver = LocalEpisodeResolver(folder_path=OUT_DIR, key_map=key_map, transform_list=None)

# LocalEpisodeResolver supports filters on episode metadata.
# It treats robot_name as a synonym of embodiment.
filters = {"robot_name": "tutorial_demo"}

resolved = resolver.resolve(filters=filters)
print("Resolved datasets:", list(resolved.keys()))

multi_from_resolver = MultiDataset._from_resolver(
    resolver=resolver,
    filters=filters,
    mode="total",
)
print("len(multi_from_resolver):", len(multi_from_resolver))
print("sample keys:", sorted(multi_from_resolver[0].keys()))

## from s3 resolver (example-only)

`S3EpisodeResolver` resolves episodes via the SQL episode table and (optionally) syncs the matching `.zarr` folders from object storage.

This is **disabled by default** so the notebook runs without credentials / `s5cmd`.


In [ ]:
RUN_S3 = True

if RUN_S3:
    # Choose a local folder where episodes should be synced.
    S3_OUT = OUT_DIR / "s3_cache"
    S3_OUT.mkdir(parents=True, exist_ok=True)

    s3 = S3EpisodeResolver(
        folder_path=S3_OUT,
        bucket_name="rldb",
        main_prefix="processed_v3",
        key_map=key_map,
        transform_list=None,
        debug=True,
    )

    # Example filters: adjust to your SQL schema/values
    filters = {
        "robot_name": "eva_bimanual",
        "is_deleted": False,
    }

    resolved = s3.resolve(filters=filters)
    print("Resolved datasets:", list(resolved.keys())[:10])

    multi_s3 = MultiDataset._from_resolver(
        resolver=s3,
        filters=filters,
        mode="percent",
        percent=0.05,
    )
    print("len(multi_s3):", len(multi_s3))
else:
    print("RUN_S3=False (skipping S3 resolver example)")